# 06 - Organizational Network Analysis
Analyze reporting structures, span of control, and department-level network metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

df = pd.read_parquet('data/processed/workforce_clean_base.parquet')
print(f'Total employees: {len(df)}')

In [ ]:
# Build reporting hierarchy graph
if 'Supervisor' in df.columns:
    G = nx.DiGraph()
    edges = df[['Supervisor', 'EmployeeID']].dropna().values
    G.add_edges_from(edges)
    print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    print(f'Isolated nodes: {nx.number_of_isolates(G)}')
    
    # Span of control
    spans = pd.Series(dict(G.out_degree())).describe()
    print(f'\nSpan of control:')
    print(f'  Mean: {spans["mean"]:.1f}')
    print(f'  Median: {spans["50%"]:.1f}')
    print(f'  Max: {spans["max"]:.0f}')

In [ ]:
# Centrality metrics
if 'Supervisor' in df.columns and G.number_of_nodes() > 1:
    # Betweenness centrality on largest weakly connected component
    wcc = max(nx.weakly_connected_components(G), key=len)
    subG = G.subgraph(wcc)
    bc = nx.betweenness_centrality(subG)
    central = pd.Series(bc).sort_values(ascending=False).head(10)
    print('Top 10 most central employees (betweenness):')
    print(central.to_string())

In [ ]:
print('\n=== Department Size Distribution ===')
print(df['DepartmentType'].value_counts().to_string())

fig, ax = plt.subplots(figsize=(8, 4))
df['DepartmentType'].value_counts().plot(kind='bar', ax=ax, title='Department Sizes')
ax.set_ylabel('Employee Count')
plt.tight_layout()